In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: Load both CSVs
df1 = pd.read_csv('../Dashboard_Final/avg_gap_scores.csv')  # contains interaction_count, etc.
df2 = pd.read_csv('../zipcodes_income.csv')  # contains Zipcodes and Income
df1['share_count'] *= 10
df1 = df1.dropna(subset=['interaction_count', 'share_count', 'population', 'program_count'])

# Step 2: Merge on zipcode (handle case differences and column names)
df2.rename(columns={'Zipcodes': 'zipcode'}, inplace=True)
df2['zipcode'] = df2['zipcode'].astype(str)
df1['zipcode'] = df1['zipcode'].astype(str)

merged_df = pd.merge(df1, df2, on='zipcode', how='inner')  # keep only matching zipcodes

# Step 3: Select features for clustering
features = merged_df[['interaction_count', 'share_count', 'population', 'program_count', 'Income']]

# Step 4: Scale features (important for clustering)
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Step 5: Choose number of clusters (try different values)
k = 3  # For example: 3 clusters
kmeans = KMeans(n_clusters=k, random_state=42)
merged_df['cluster'] = kmeans.fit_predict(features_scaled)

# Step 6: Visualize clusters (pairplot)
sns.pairplot(merged_df, vars=['interaction_count', 'share_count', 'population', 'program_count', 'Income'], 
             hue='cluster', palette='tab10')
plt.suptitle('Clusters of Zipcodes Based on Features', y=1.02)
plt.show()






In [ ]:
# Reverse scaling for interpretability
centroids = scaler.inverse_transform(kmeans.cluster_centers_)
centroid_df = pd.DataFrame(centroids, columns=features.columns)
centroid_df['cluster'] = range(k)
print(centroid_df)


In [ ]:
summary = merged_df.groupby('cluster')[['interaction_count', 'share_count', 'population', 'program_count', 'Income']].mean()
print(summary)
